In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import warnings
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score




In [ ]:
import kagglehub
from sklearn.preprocessing import StandardScaler


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# reading the csv file from path
# i hope file name is corectly written
df = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
# showing the first rows to check data
# looks good i think
df.head()

In [ ]:
# Task 3: Write your code here:
# geting the information about columns and nulls
# i will use info function
df.info()

In [ ]:
# Task 4: Write your code here:
# show statical description for the data
# describe gives mean and std and other things
df.describe()

In [ ]:
# Task 1: Write your code here:
# checking for any missing values first
# then i will drop them or fill, maybe drop is faster
print(df.isnull().sum())
df = df.dropna()

In [ ]:
# Task 2: Write your code here:
# looking for duplicate rows in the data
# i dont want same info twice
print("duplicated rows count:", df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:
# change text columns to numbers so model can understand
# i will use get_dummies for simple encoding
df = pd.get_dummies(df)

In [ ]:
# Task 4: Write your code here:
# making all numerical values in same scale
# i useing StandardScaler as requested
scaler = StandardScaler()

# i will separate target before scaling
# assume target column name is 'target' or 'default' (check your data info)
# for now i scale everything except target if i know name
# lets say we scale all numerical cols
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
# Task 5: Write your code here:
# see if classes are equall or not
# very importent to check imbalance
target_counts = df['target'].value_counts() # change 'target' to actual label name
print(target_counts)

if target_counts.iloc[0] / target_counts.iloc[1] > 2:
    print("the data is imbalanced!")
else:
    print("the data looks okey")

In [ ]:
# Task 1: Write your code here:
# separating the target from features
# target is 'target' col, and X is evrything else
X = df.drop('target', axis=1) # make sure 'target' is the right name
y = df['target']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
# i will use StratifiedKFold because its better for imbalance data
# and i will train CatBoost model inside the loop
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

# lets start the cross val now
for train_index, test_index in skf.split(X, y):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    # define the catboost model with some params
    # silent=True so it dont print too much logs
    model = CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, silent=True)

    # fiting the model
    model.fit(X_train_fold, y_train_fold)

    # making prediction to evaluate
    preds = model.predict(X_test_fold)

    # i use F1 Score becuase accuracy is not good for imbalanced
    fold_f1 = f1_score(y_test_fold, preds)
    scores.append(fold_f1)
    print(f"one fold F1 Score: {fold_f1:.4f}")

# finaly print the average score for all folds
print("\n--- Final Results ---")
print("Average F1 Score across all folds:", np.mean(scores))

In [ ]:
# Task 1: Write your code here:
# checking wich feature is the most importent
# i will use the feature_importances_ from our catboost model
feat_importances = model.get_feature_importance()
feature_names = X.columns

# ploting it to see clearly
plt.figure(figsize=(12, 8))
plt.barh(feature_names, feat_importances)
plt.title("Wich feature is the best?")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

In [ ]:
# Task 2: Write your code here:
# finding the name of the feature with highest score
# i call it the golden feature as requested
golden_index = np.argmax(feat_importances)
golden_feature_name = feature_names[golden_index]

print("--- The Resault ---")
print("The Golden Feature is:", golden_feature_name)
print("This is the most powerfull predictor in the data!")

In [ ]:
# Task Bonus: Write your code here:
# now i will try to train the model with ONLY the best feature
# i want to see if one feature is enough to get good resault
# creating new X with just one col
X_golden = X[[golden_feature_name]]

# i use same loop and same skf for fair comparison
golden_scores = []

print("starting training with golden feature only...")

for train_idx, test_idx in skf.split(X_golden, y):
    X_train_g, X_test_g = X_golden.iloc[train_idx], X_golden.iloc[test_idx]
    y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

    # same catboost model
    model_g = CatBoostClassifier(iterations=100, silent=True)
    model_g.fit(X_train_g, y_train_g)

    # predict and calculate accuracy this time for compare
    g_preds = model_g.predict(X_test_g)
    from sklearn.metrics import accuracy_score
    acc = accuracy_score(y_test_g, g_preds)
    golden_scores.append(acc)

# printing and comparing
avg_golden_acc = np.mean(golden_scores)
print(f"\nAccuracy with ONLY Golden Feature: {avg_golden_acc:.4f}")

